## 6.2 Simple RNN反向传播 - 梯度推导

#### 1. 这一节我们要解决什么问题 🎯

前面我们已经理解了 RNN 反向传播的整体流程，现在这一节开始正式推导梯度。

这一节主要解决三个问题：

* 输出层的梯度怎么求
* 隐藏状态的梯度怎么求
* 隐藏层参数的梯度怎么求

这一节最核心的思路是：

> 先从当前时间步的输出层开始反传，再把梯度传回隐藏状态，最后继续传到隐藏层参数。


#### 2. 先回顾前向传播公式 🧠

在第 $t$ 个时间步，前向传播写成：

##### 2.1 隐藏层部分

$ a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h $

$ h_t = f(a_t) $

##### 2.2 输出层部分

$ o_t = W_{hy}h_t + b_y $

$ \hat{y}_t = g(o_t) $

##### 2.3 总损失

$ L = \sum_{t=1}^{T} L_t $

#### 3. 输出层梯度推导 📤

##### 3.1 定义输出层误差项

先定义：

$ \delta_t^{(o)} = \frac{\partial L_t}{\partial o_t} $

它表示：

> 第 $t$ 个时间步损失，对输出层线性结果 $o_t$ 的梯度。

如果是常见的 $softmax + cross\ entropy$，则：

$ \delta_t^{(o)} = \hat{y}_t - y_t $

##### 3.2 输出层参数梯度

因为：

$ o_t = W_{hy}h_t + b_y $

所以第 $t$ 个时间步对输出层参数的梯度贡献为：

$ \frac{\partial L_t}{\partial W_{hy}} = \delta_t^{(o)} h_t^T $

$ \frac{\partial L_t}{\partial b_y} = \delta_t^{(o)} $

由于参数在所有时间步共享，所以整个序列总梯度为：

$ \frac{\partial L}{\partial W_{hy}} = \sum_{t=1}^{T} \delta_t^{(o)} h_t^T $

$ \frac{\partial L}{\partial b_y} = \sum_{t=1}^{T} \delta_t^{(o)} $

##### 3.3 输出层传回隐藏状态的梯度

输出层还要继续把梯度传回隐藏状态：

$ \frac{\partial L_t}{\partial h_t}\Big|_{\text{output}} = W_{hy}^T \delta_t^{(o)} $

这表示：

> 当前输出层传回给当前隐藏状态的梯度。

#### 4. 隐藏状态梯度推导 🔁

##### 4.1 为什么隐藏状态梯度更特殊？

因为 $h_t$ 不仅影响当前输出层，还会影响下一时刻隐藏状态。

所以 $h_t$ 的梯度有两个来源：

* 当前输出层传回来的梯度
* 下一时刻沿时间链传回来的梯度

因此，总损失对 $h_t$ 的梯度为：

$ \frac{\partial L}{\partial h_t} = \frac{\partial L_t}{\partial h_t}\Big|_{\text{output}} + \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t} $

这里第二项表示：

> 未来时间步通过隐藏状态链传回来的梯度。

##### 4.2 定义隐藏层误差项

定义：

$ \delta_t^{(h)} = \frac{\partial L}{\partial a_t} $

它表示：

> 总损失对隐藏层线性结果 $a_t$ 的梯度。

因为：

$ h_t = f(a_t) $

所以根据链式法则：

$ \delta_t^{(h)} = \frac{\partial L}{\partial h_t} \odot f'(a_t) $

把前面的结果代入：

$ \delta_t^{(h)} = \left( W_{hy}^T \delta_t^{(o)} + W_{hh}^T \delta_{t+1}^{(h)} \right) \odot f'(a_t) $

这就是 RNN 反向传播中最核心的递推公式。

##### 4.3 如果激活函数是 $tanh$

因为：

$ f(a_t) = tanh(a_t) $

所以：

$ f'(a_t) = 1 - h_t^2 $

于是隐藏层误差项也可以写成：

$ \delta_t^{(h)} = \left( W_{hy}^T \delta_t^{(o)} + W_{hh}^T \delta_{t+1}^{(h)} \right) \odot (1 - h_t^2) $


#### 5. 隐藏层参数梯度推导 📦

##### 5.1 输入到隐藏层权重 $W_{xh}$

因为：

$ a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h $

所以第 $t$ 个时间步对 $W_{xh}$ 的梯度贡献为：

$ \frac{\partial L_t}{\partial W_{xh}} = \delta_t^{(h)} x_t^T $

整个序列总梯度为：

$ \frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^{T} \delta_t^{(h)} x_t^T $

##### 5.2 隐藏到隐藏层权重 $W_{hh}$

同理：

$ \frac{\partial L_t}{\partial W_{hh}} = \delta_t^{(h)} h_{t-1}^T $

整个序列总梯度为：

$ \frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t^{(h)} h_{t-1}^T $

##### 5.3 隐藏层偏置 $b_h$

因为偏置直接加到 $a_t$ 上，所以：

$ \frac{\partial L_t}{\partial b_h} = \delta_t^{(h)} $

整个序列总梯度为：

$ \frac{\partial L}{\partial b_h} = \sum_{t=1}^{T} \delta_t^{(h)} $

#### 6. 本节总结 🧾

这一节最核心的推导逻辑可以概括为：

* 先定义输出层误差项  
  $ \delta_t^{(o)} = \frac{\partial L_t}{\partial o_t} $
* 再由输出层把梯度传回隐藏状态
* 然后结合未来时间步传回来的梯度，得到隐藏层误差项  
  $ \delta_t^{(h)} = \frac{\partial L}{\partial a_t} $
* 最后利用 $ \delta_t^{(h)} $ 推出隐藏层参数梯度

所以整个梯度推导的主线是：

> 输出层误差 $\rightarrow$ 隐藏状态梯度 $\rightarrow$ 隐藏层误差项 $\rightarrow$ 参数梯度